# Build arXiv PDF Text Corpus (PyMuPDF + pypdf)

Notebook này mở rộng pipeline benchmark theo hướng gần thực tế hơn:

- Crawl/tải PDF từ **arXiv** theo **cùng domain**
- Parse mỗi PDF bằng nhiều parser text-based:
  - **PyMuPDF**
  - **pypdf**

- Tạo benchmark:
  - **positive pairs**: cùng tài liệu, khác parser/OCR method
  - **hard negatives**: khác tài liệu nhưng cùng domain
  - **online stream benchmark**



## 0. Cài thư viện cơ bản

Cell này cài các thư viện nhẹ để crawl, parse text-based và render PDF.


In [1]:
%pip -q install requests feedparser pymupdf pypdf tqdm pandas


Note: you may need to restart the kernel to use updated packages.


## 1. Cài Tesseract hệ thống 


In [2]:
# Không cần cài Tesseract vì chỉ chạy PyMuPDF và pypdf
# !apt-get update -y
# !apt-get install -y tesseract-ocr poppler-utils


## 2. Import và cấu hình

In [3]:
from pathlib import Path
import re
import os
import json
import time
import math
import hashlib
import random
import requests
import feedparser
import pandas as pd
from tqdm.auto import tqdm
from itertools import combinations

random.seed(42)

BASE_DIR = Path("arxiv_ocr_benchmark_workspace")
PDF_DIR = BASE_DIR / "pdfs"
META_DIR = BASE_DIR / "metadata"
PARSED_DIR = BASE_DIR / "parsed_jsonl"
BENCH_DIR = BASE_DIR / "benchmark"

for d in [PDF_DIR, META_DIR, PARSED_DIR, BENCH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

BASE_DIR


c:\Users\pntha\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WindowsPath('arxiv_ocr_benchmark_workspace')

## 3. arXiv API search

Mặc định nên dùng **một category/domain** để hard negatives khó hơn.


In [4]:
ARXIV_API_URL = "http://export.arxiv.org/api/query"

def search_arxiv(query: str, start: int = 0, max_results: int = 20, sort_by: str = "submittedDate", sort_order: str = "descending"):
    params = {
        "search_query": query,
        "start": start,
        "max_results": max_results,
        "sortBy": sort_by,
        "sortOrder": sort_order,
    }
    r = requests.get(ARXIV_API_URL, params=params, timeout=60)
    r.raise_for_status()
    feed = feedparser.parse(r.text)

    rows = []
    for entry in feed.entries:
        pdf_url = None
        for link in entry.get("links", []):
            if link.get("type") == "application/pdf":
                pdf_url = link.get("href")
                break

        rows.append({
            "id": entry.get("id"),
            "arxiv_id": entry.get("id", "").rstrip("/").split("/")[-1],
            "title": re.sub(r"\s+", " ", entry.get("title", "")).strip(),
            "summary": re.sub(r"\s+", " ", entry.get("summary", "")).strip(),
            "published": entry.get("published"),
            "updated": entry.get("updated"),
            "authors": [a.get("name") for a in entry.get("authors", [])],
            "pdf_url": pdf_url,
            "primary_category": entry.get("arxiv_primary_category", {}).get("term"),
            "categories": [t["term"] for t in entry.get("tags", [])] if entry.get("tags") else [],
        })
    return pd.DataFrame(rows)

In [5]:
QUERIES = [
    "cat:cs.CL AND all:question answering",
    "cat:cs.CL AND all:information extraction",
    "cat:cs.CL AND all:summarization",
    "cat:cs.CL AND all:machine translation",
    "cat:cs.CL AND all:retrieval augmented generation",

    "cat:cs.IR AND all:retrieval",
    "cat:cs.IR AND all:reranking",
    "cat:cs.IR AND all:dense retrieval",
    "cat:cs.IR AND all:document ranking",
    "cat:cs.IR AND all:search",

    "cat:cs.LG AND all:contrastive learning",
    "cat:cs.LG AND all:representation learning",
    "cat:cs.LG AND all:domain adaptation",
    "cat:cs.LG AND all:self supervised learning",

    "cat:cs.AI AND all:knowledge graphs",
    "cat:cs.AI AND all:reasoning",
    "cat:cs.AI AND all:planning",
    "cat:cs.AI AND all:agents",
]
MAX_RESULTS_PER_QUERY = 150

all_meta = []
for query in QUERIES:
    df_q = search_arxiv(query=query, max_results=MAX_RESULTS_PER_QUERY)
    df_q["requested_query"] = query
    all_meta.append(df_q)

df = (
    pd.concat(all_meta, ignore_index=True)
    .drop_duplicates(subset=["arxiv_id"])
    .reset_index(drop=True)
)

print("Số query:", len(QUERIES))
print("Số bài sau khi gộp và bỏ trùng:", len(df))
df[["arxiv_id", "title", "primary_category", "requested_query", "published"]].head(10)

Số query: 18
Số bài sau khi gộp và bỏ trùng: 1579


,arxiv_id,title,primary_category,requested_query,published
0,2604.01212v1,$\texttt{YC-Bench}$: Benchmarking AI Agents fo...,cs.CL,cat:cs.CL AND all:question answering,2026-04-01T17:52:19Z
1,2604.01195v1,ORBIT: Scalable and Verifiable Data Generation...,cs.CL,cat:cs.CL AND all:question answering,2026-04-01T17:42:41Z
2,2604.01193v1,Embarrassingly Simple Self-Distillation Improv...,cs.CL,cat:cs.CL AND all:question answering,2026-04-01T17:39:50Z
3,2604.01181v1,True (VIS) Lies: Analyzing How Generative AI R...,cs.HC,cat:cs.CL AND all:question answering,2026-04-01T17:33:10Z
4,2604.01029v1,Revision or Re-Solving? Decomposing Second-Pas...,cs.SE,cat:cs.CL AND all:question answering,2026-04-01T15:39:40Z
5,2604.00986v1,Do Phone-Use Agents Respect Your Privacy?,cs.CR,cat:cs.CL AND all:question answering,2026-04-01T14:50:50Z
6,2604.00913v1,Benchmarking and Mechanistic Analysis of Visio...,cs.CV,cat:cs.CL AND all:question answering,2026-04-01T13:55:28Z
7,2604.00890v1,Beyond Symbolic Solving: Multi Chain-of-Though...,cs.AI,cat:cs.CL AND all:question answering,2026-04-01T13:37:06Z
8,2604.00778v1,From Early Encoding to Late Suppression: Inter...,cs.CL,cat:cs.CL AND all:question answering,2026-04-01T11:40:12Z
9,2604.00698v1,Learning to Hint for Reinforcement Learning,cs.LG,cat:cs.CL AND all:question answering,2026-04-01T09:58:08Z


In [6]:
meta_csv_path = META_DIR / "arxiv_search_results_multi_query.csv"
meta_jsonl_path = META_DIR / "arxiv_search_results_multi_query.jsonl"


df.to_csv(meta_csv_path, index=False, encoding="utf-8")

with open(meta_jsonl_path, "w", encoding="utf-8") as f:
    for row in df.to_dict(orient="records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(meta_csv_path)
print(meta_jsonl_path)

arxiv_ocr_benchmark_workspace\metadata\arxiv_search_results_multi_query.csv
arxiv_ocr_benchmark_workspace\metadata\arxiv_search_results_multi_query.jsonl


In [7]:
len(df)

1579

## 4. Tải PDF

In [8]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def safe_filename(text: str) -> str:
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text)
    return text[:180]

def download_pdf(row, pdf_dir: Path, sleep_sec: float = 0.5, overwrite: bool = False):
    arxiv_id = row["arxiv_id"]
    pdf_url = row["pdf_url"]

    if not pdf_url:
        return {
            "arxiv_id": arxiv_id,
            "pdf_path": None,
            "ok": False,
            "error": "missing_pdf_url",
        }

    out_path = pdf_dir / f"{safe_filename(arxiv_id)}.pdf"

    if out_path.exists() and not overwrite:
        return {
            "arxiv_id": arxiv_id,
            "pdf_path": str(out_path),
            "ok": True,
            "cached": True,
        }

    r = requests.get(pdf_url, timeout=120)
    r.raise_for_status()
    out_path.write_bytes(r.content)
    time.sleep(sleep_sec)

    return {
        "arxiv_id": arxiv_id,
        "pdf_path": str(out_path),
        "ok": True,
        "cached": False,
    }

def download_pdfs_in_batches(
    df,
    pdf_dir: Path,
    batch_size: int = 50,
    max_workers: int = 8,
    sleep_sec: float = 0.5,
    overwrite: bool = False,
):
    rows = df.to_dict(orient="records")
    results = []

    for start in range(0, len(rows), batch_size):
        batch_rows = rows[start:start + batch_size]

        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            futures = {
                ex.submit(download_pdf, row, pdf_dir, sleep_sec, overwrite): row
                for row in batch_rows
            }

            for fut in tqdm(as_completed(futures), total=len(futures), desc=f"Batch {start//batch_size + 1}"):
                try:
                    results.append(fut.result())
                except Exception as e:
                    row = futures[fut]
                    results.append({
                        "arxiv_id": row.get("arxiv_id"),
                        "pdf_path": None,
                        "ok": False,
                        "error": str(e),
                    })

        time.sleep(2.0)

    return pd.DataFrame(results)

download_df = download_pdfs_in_batches(
    df=df,
    pdf_dir=PDF_DIR,
    batch_size=50,
    max_workers=8,
    sleep_sec=0.5,
    overwrite=False,
)

download_df.head()

Batch 32: 100%|██████████| 29/29 [00:10<00:00,  2.71it/s]


,arxiv_id,pdf_path,ok,cached,error
0,2604.00890v1,arxiv_ocr_benchmark_workspace\pdfs\2604.00890v...,True,False,NaN
1,2604.00913v1,arxiv_ocr_benchmark_workspace\pdfs\2604.00913v...,True,False,NaN
2,2604.01195v1,arxiv_ocr_benchmark_workspace\pdfs\2604.01195v...,True,False,NaN
3,2604.01212v1,arxiv_ocr_benchmark_workspace\pdfs\2604.01212v...,True,False,NaN
4,2604.01029v1,arxiv_ocr_benchmark_workspace\pdfs\2604.01029v...,True,False,NaN


In [9]:
print("Số file PDF trong thư mục:", len(list(PDF_DIR.glob("*.pdf"))))

Số file PDF trong thư mục: 1578


## 5. Text-based parsers

Ba parser này là baseline nhẹ, nhanh.


In [10]:
import fitz
from pypdf import PdfReader

def extract_with_pymupdf(pdf_path: str):
    doc = fitz.open(pdf_path)
    pages = []
    for i, page in enumerate(doc):
        txt = page.get_text("text")
        pages.append({"page_num": i + 1, "text": txt})
    return {
        "parser": "pymupdf",
        "page_count": len(pages),
        "pages": pages,
        "text": "\n\n".join(p["text"] for p in pages),
    }

def extract_with_pypdf(pdf_path: str):
    reader = PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages):
        try:
            txt = page.extract_text() or ""
        except Exception:
            txt = ""
        pages.append({"page_num": i + 1, "text": txt})
    return {
        "parser": "pypdf",
        "page_count": len(pages),
        "pages": pages,
        "text": "\n\n".join(p["text"] for p in pages),
    }


## 10. Chuẩn hóa và đóng gói output

In [11]:
def normalize_text(text: str) -> str:
    if text is None:
        return ""
    text = text.replace("\x00", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def sha1_text(text: str) -> str:
    return hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()

def build_record(meta_row: dict, pdf_path: str, parsed: dict):
    text = normalize_text(parsed["text"])
    return {
        "doc_id": meta_row["arxiv_id"],
        "source_pdf": str(pdf_path),
        "parser_name": parsed["parser"],
        "text": text,
        "text_sha1": sha1_text(text),
        "char_count": len(text),
        "page_count": parsed["page_count"],
        "pages": parsed["pages"],
        "metadata": {
            "title": meta_row["title"],
            "summary": meta_row["summary"],
            "published": meta_row["published"],
            "updated": meta_row["updated"],
            "authors": meta_row["authors"],
            "pdf_url": meta_row["pdf_url"],
            "primary_category": meta_row["primary_category"],
            "categories": meta_row["categories"],
            "source": "arxiv",
        }
    }

## 11. Chọn parser nào sẽ chạy thực tế

Mặc định chạy:
- **PyMuPDF**
- **pypdf**



In [13]:
from pathlib import Path

PARSERS = {
    "pymupdf": extract_with_pymupdf,
    "pypdf": extract_with_pypdf,
}

if RUN_TESSERACT:
    PARSERS["tesseract_ocr"] = lambda pdf_path: extract_with_tesseract(
        pdf_path=pdf_path,
        image_dir=IMG_DIR,
        dpi=OCR_DPI,
        max_pages=OCR_MAX_PAGES,
        lang=OCR_LANG,
    )

if RUN_LIGHTON:
    PARSERS["lightonocr_2_1b"] = lambda pdf_path: extract_with_lightonocr(
        pdf_path=pdf_path,
        image_dir=IMG_DIR,
        dpi=OCR_DPI,
        max_pages=OCR_MAX_PAGES,
        batch_size=OCR_MODEL_BATCH_SIZE,
    )

if RUN_QWEN_OCR:
    PARSERS["qwen2_5_vl_7b_ocr"] = lambda pdf_path: extract_with_qwen_ocr(
        pdf_path=pdf_path,
        image_dir=IMG_DIR,
        dpi=OCR_DPI,
        max_pages=OCR_MAX_PAGES,
        batch_size=OCR_MODEL_BATCH_SIZE,
    )

print("Parsers sẽ chạy:", list(PARSERS.keys()))


Parsers sẽ chạy: ['pymupdf', 'pypdf']


## 12. Parse toàn bộ PDF bằng các parser đã chọn

In [14]:
records = []
errors = []

meta_by_id = {row["arxiv_id"]: row for _, row in df.iterrows()}

for pdf_path in tqdm(sorted(PDF_DIR.glob("*.pdf"))):
    arxiv_id = pdf_path.stem
    if arxiv_id not in meta_by_id:
        continue

    meta_row = meta_by_id[arxiv_id]
    for parser_name, parser_fn in PARSERS.items():
        try:
            parsed = parser_fn(str(pdf_path))
            rec = build_record(meta_row, str(pdf_path), parsed)
            records.append(rec)
        except Exception as e:
            errors.append({
                "doc_id": arxiv_id,
                "parser_name": parser_name,
                "pdf_path": str(pdf_path),
                "error": str(e),
            })

print("Số parser outputs:", len(records))
print("Số lỗi parse:", len(errors))

 12%|█▏        | 184/1578 [03:04<34:22,  1.48s/it] Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)
Ignoring wrong pointing object 37 0 (offset 0)
Ignoring wrong pointing object 39 0 (offset 0)
Ignoring wrong pointing object 54 0 (offset 0)
Ignoring wrong pointing object 59 0 (offset 0)
Ignoring wrong pointing object 61 0 (offset 0)
Ignoring wrong pointing object 73 0 (offset 0)
Ignoring wrong pointing object 75 0 (offset 0)
Ignoring wrong pointing object 119 0 (offset 0)
Ignoring wrong pointing object 145 0 (offset 0)
Ignoring wrong pointing object 150 0 (offset 0)
 17%|█▋        | 262/1578 [04:37<33:34,  1.53s/it]  Ignoring wrong pointing object 10 0 (offset 0)
Ign

MuPDF error: syntax error: unknown keyword: 'minorversion'



 57%|█████▋    | 895/1578 [19:44<20:21,  1.79s/it]  Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 23 0 (offset 0)
Ignoring wrong pointing object 25 0 (offset 0)
Ignoring wrong pointing object 27 0 (offset 0)
Ignoring wrong pointing object 29 0 (offset 0)
Ignoring wrong pointing object 37 0 (offset 0)
Ignoring wrong pointing object 39 0 (offset 0)
Ignoring wrong pointing object 41 0 (offset 0)
Ignoring wrong pointing object 43 0 (offset 0)
Ignoring wrong pointing object 55 0 (offset 0)
Ignoring wrong pointing object 62 0 (offset 0)
Ignoring wrong pointing object 94 0 (offset 0)
 59%|█████▊    | 925/1578 [20:08<09:55,  1.10it/s]Ignoring wrong pointing object 6 0 (offset 0)
Ignoring

MuPDF error: format error: cmsOpenProfileFromMem failed



 70%|███████   | 1105/1578 [23:09<08:08,  1.03s/it]  Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 19 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 33 0 (offset 0)
Ignoring wrong pointing object 35 0 (offset 0)
Ignoring wrong pointing object 38 0 (offset 0)
Ignoring wrong pointing object 53 0 (offset 0)
Ignoring wrong pointing object 55 0 (offset 0)
Ignoring wrong pointing object 60 0 (offset 0)
Ignoring wrong pointing object 62 0 (offset 0)
Ignoring wrong pointing object 64 0 (offset 0)
Ignoring wrong pointing object 70 0 (offset 0)
Ignoring wrong pointing object 93 0 (offset 0)
 76%|███████▋  | 1207/1578 [26:24<26:07,  4.22s/it]Ignoring wrong pointing object 6 0 (offset 0)
Ignori

Số parser outputs: 3156
Số lỗi parse: 0


In [15]:
parsed_df = pd.DataFrame([
    {
        "doc_id": r["doc_id"],
        "parser_name": r["parser_name"],
        "char_count": r["char_count"],
        "page_count": r["page_count"],
        "text_sha1": r["text_sha1"],
        "primary_category": r["metadata"]["primary_category"],
        "title": r["metadata"]["title"],
    }
    for r in records
])

parsed_df.head(20)

,doc_id,parser_name,char_count,page_count,text_sha1,primary_category,title
0,2507.21563v3,pymupdf,66139,15,99af3a19398330797fa6a712f4ccbe032bbadb11,cs.IR,VoteGCL: Enhancing Graph-based Recommendations...
1,2507.21563v3,pypdf,70498,15,9b914e7416830bcb4a2345dad3ad46e569b889d9,cs.IR,VoteGCL: Enhancing Graph-based Recommendations...
2,2508.01514v1,pymupdf,39835,7,802c84aa6a58e082118c6ff9c995153ddfd42490,cs.IR,End-to-End Personalization: Unifying Recommend...
3,2508.01514v1,pypdf,39857,7,8a5f290830d268aa011a0dd601e80634ccc842c5,cs.IR,End-to-End Personalization: Unifying Recommend...
4,2508.05512v1,pymupdf,31570,6,c78e1930a7b0d1c1a308c4ed041c5c0c9afb0983,cs.IR,RankArena: A Unified Platform for Evaluating R...
5,2508.05512v1,pypdf,31523,6,2ea9ee4d8816d9d4dc46e4f5d6dd4a600ee3753e,cs.IR,RankArena: A Unified Platform for Evaluating R...
6,2508.05664v1,pymupdf,14057,6,e7104b40189016cfe2c4ea4ce5580180778b3676,cs.IR,Enhancing Retrieval-Augmented Generation for E...
7,2508.05664v1,pypdf,13993,6,6b266312d01e89a0a71ca7633f92b0796364f98e,cs.IR,Enhancing Retrieval-Augmented Generation for E...
8,2508.07050v2,pymupdf,97162,21,f3914803ed91bc01f9c7b9a28607cf72ff79d489,cs.IR,ReasonRank: Empowering Passage Ranking with St...
9,2508.07050v2,pypdf,97256,21,c3f0f975e9d0a414d0c528a6c367b026ecbab069,cs.IR,ReasonRank: Empowering Passage Ranking with St...


## 13. Xuất JSONL parser outputs

In [17]:
all_jsonl_path = PARSED_DIR / "arxiv_multi_parser_ocr_all.jsonl"
with open(all_jsonl_path, "w", encoding="utf-8", errors='replace') as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(all_jsonl_path)

for parser_name in sorted(set(r["parser_name"] for r in records)):
    parser_path = PARSED_DIR / f"{parser_name}.jsonl"
    with open(parser_path, "w", encoding="utf-8", errors='replace') as f:
        for r in records:
            if r["parser_name"] == parser_name:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(parser_path)


arxiv_ocr_benchmark_workspace\parsed_jsonl\arxiv_multi_parser_ocr_all.jsonl
arxiv_ocr_benchmark_workspace\parsed_jsonl\pymupdf.jsonl
arxiv_ocr_benchmark_workspace\parsed_jsonl\pypdf.jsonl


## 14. Kiểm tra cùng document có ra nhiều version text khác nhau không

In [18]:
summary = (
    parsed_df.groupby("doc_id")
    .agg(
        n_parsers=("parser_name", "nunique"),
        n_unique_texts=("text_sha1", "nunique"),
        min_chars=("char_count", "min"),
        max_chars=("char_count", "max"),
    )
    .reset_index()
    .sort_values(["n_unique_texts", "max_chars"], ascending=[False, False])
)

summary.head(20)

,doc_id,n_parsers,n_unique_texts,min_chars,max_chars
508,2603.20179v2,2,2,924182,928306
751,2603.25100v1,2,2,681628,682449
545,2603.21152v3,2,2,437319,509082
796,2603.25681v1,2,2,469528,471820
254,2602.20507v1,2,2,421194,421568
741,2603.24955v1,2,2,347119,348733
805,2603.25862v1,2,2,329282,329897
1394,2604.00324v1,2,2,317547,318103
738,2603.24917v1,2,2,313538,316984
127,2601.14004v3,2,2,310624,312266


## 15. Tạo positive pairs

Positive:
- cùng `doc_id`
- khác `parser_name`

Điều này cho phép positive được sinh cả từ:
- parser text-based khác nhau
- OCR vs non-OCR
- model OCR khác nhau


In [19]:
mini_records = [
    {
        "doc_id": r["doc_id"],
        "parser_name": r["parser_name"],
        "text_sha1": r["text_sha1"],
        "char_count": r["char_count"],
        "primary_category": r["metadata"]["primary_category"],
        "title": r["metadata"]["title"],
    }
    for r in records
]

by_doc = {}
for r in mini_records:
    by_doc.setdefault(r["doc_id"], []).append(r)

positive_pairs = []
for doc_id, items in by_doc.items():
    for a, b in combinations(items, 2):
        positive_pairs.append({
            "left_doc_id": a["doc_id"],
            "left_parser": a["parser_name"],
            "right_doc_id": b["doc_id"],
            "right_parser": b["parser_name"],
            "left_category": a["primary_category"],
            "right_category": b["primary_category"],
            "label": 1,
            "pair_type": "same_doc_diff_parser",
        })

positive_pairs_df = pd.DataFrame(positive_pairs)
print("Số positive pairs:", len(positive_pairs_df))
positive_pairs_df.head(20)

Số positive pairs: 1578


,left_doc_id,left_parser,right_doc_id,right_parser,left_category,right_category,label,pair_type
0,2507.21563v3,pymupdf,2507.21563v3,pypdf,cs.IR,cs.IR,1,same_doc_diff_parser
1,2508.01514v1,pymupdf,2508.01514v1,pypdf,cs.IR,cs.IR,1,same_doc_diff_parser
2,2508.05512v1,pymupdf,2508.05512v1,pypdf,cs.IR,cs.IR,1,same_doc_diff_parser
3,2508.05664v1,pymupdf,2508.05664v1,pypdf,cs.IR,cs.IR,1,same_doc_diff_parser
4,2508.07050v2,pymupdf,2508.07050v2,pypdf,cs.IR,cs.IR,1,same_doc_diff_parser
5,2508.07995v4,pymupdf,2508.07995v4,pypdf,cs.IR,cs.IR,1,same_doc_diff_parser
6,2508.10068v2,pymupdf,2508.10068v2,pypdf,cs.SE,cs.SE,1,same_doc_diff_parser
7,2508.11670v1,pymupdf,2508.11670v1,pypdf,cs.IR,cs.IR,1,same_doc_diff_parser
8,2508.13517v1,pymupdf,2508.13517v1,pypdf,cs.IR,cs.IR,1,same_doc_diff_parser
9,2508.13889v2,pymupdf,2508.13889v2,pypdf,cs.IR,cs.IR,1,same_doc_diff_parser


## 16. Tạo hard negative pairs cùng domain

Hard negative:
- khác `doc_id`
- cùng `primary_category`
- mặc định lấy từ **cùng parser** để benchmark chặt hơn


In [20]:
def build_hard_negatives(records, negatives_per_doc=4, parser_name="pymupdf"):
    pool = [r for r in records if r["parser_name"] == parser_name]
    by_cat = {}
    for r in pool:
        by_cat.setdefault(r["primary_category"], []).append(r)

    negatives = []
    seen = set()

    for cat, items in by_cat.items():
        for a in items:
            candidates = [x for x in items if x["doc_id"] != a["doc_id"]]
            random.shuffle(candidates)
            picked = 0
            for b in candidates:
                key = tuple(sorted([(a["doc_id"], a["parser_name"]), (b["doc_id"], b["parser_name"])]))
                if key in seen:
                    continue
                seen.add(key)
                negatives.append({
                    "left_doc_id": a["doc_id"],
                    "left_parser": a["parser_name"],
                    "right_doc_id": b["doc_id"],
                    "right_parser": b["parser_name"],
                    "left_category": a["primary_category"],
                    "right_category": b["primary_category"],
                    "label": 0,
                    "pair_type": "same_domain_hard_negative",
                })
                picked += 1
                if picked >= negatives_per_doc:
                    break
    return pd.DataFrame(negatives)

NEGATIVE_PARSER = "pymupdf"  # có thể đổi sang "tesseract_ocr" hoặc parser khác
hard_negative_df = build_hard_negatives(mini_records, negatives_per_doc=4, parser_name=NEGATIVE_PARSER)
print("Số hard negative pairs:", len(hard_negative_df))
hard_negative_df.head(20)

Số hard negative pairs: 6080


,left_doc_id,left_parser,right_doc_id,right_parser,left_category,right_category,label,pair_type
0,2507.21563v3,pymupdf,2603.22344v1,pymupdf,cs.IR,cs.IR,0,same_domain_hard_negative
1,2507.21563v3,pymupdf,2602.05334v1,pymupdf,cs.IR,cs.IR,0,same_domain_hard_negative
2,2507.21563v3,pymupdf,2603.22340v1,pymupdf,cs.IR,cs.IR,0,same_domain_hard_negative
3,2507.21563v3,pymupdf,2603.20338v1,pymupdf,cs.IR,cs.IR,0,same_domain_hard_negative
4,2508.01514v1,pymupdf,2602.12510v1,pymupdf,cs.IR,cs.IR,0,same_domain_hard_negative
5,2508.01514v1,pymupdf,2603.13301v1,pymupdf,cs.IR,cs.IR,0,same_domain_hard_negative
6,2508.01514v1,pymupdf,2603.22335v1,pymupdf,cs.IR,cs.IR,0,same_domain_hard_negative
7,2508.01514v1,pymupdf,2603.07179v1,pymupdf,cs.IR,cs.IR,0,same_domain_hard_negative
8,2508.05512v1,pymupdf,2603.14170v1,pymupdf,cs.IR,cs.IR,0,same_domain_hard_negative
9,2508.05512v1,pymupdf,2602.16964v1,pymupdf,cs.IR,cs.IR,0,same_domain_hard_negative


## 17. Ghép pair benchmark và xuất file

In [21]:
pair_benchmark_df = pd.concat(
    [positive_pairs_df, hard_negative_df],
    ignore_index=True
).sample(frac=1.0, random_state=42).reset_index(drop=True)

pair_csv = BENCH_DIR / "same_domain_pair_benchmark_with_ocr.csv"
pair_jsonl = BENCH_DIR / "same_domain_pair_benchmark_with_ocr.jsonl"

pair_benchmark_df.to_csv(pair_csv, index=False, encoding="utf-8")
with open(pair_jsonl, "w", encoding="utf-8") as f:
    for row in pair_benchmark_df.to_dict(orient="records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(pair_benchmark_df["label"].value_counts(dropna=False))
print(pair_csv)
print(pair_jsonl)

label
0    6080
1    1578
Name: count, dtype: int64
arxiv_ocr_benchmark_workspace\benchmark\same_domain_pair_benchmark_with_ocr.csv
arxiv_ocr_benchmark_workspace\benchmark\same_domain_pair_benchmark_with_ocr.jsonl


## 18. Online stream benchmark

Logic online:
- lần đầu một `doc_id` xuất hiện -> label 0
- các lần sau của cùng `doc_id` -> label 1


In [22]:
stream_df = pd.DataFrame(mini_records).sample(frac=1.0, random_state=42).reset_index(drop=True)

seen = set()
labels = []
for _, row in stream_df.iterrows():
    is_dup = 1 if row["doc_id"] in seen else 0
    labels.append(is_dup)
    seen.add(row["doc_id"])

stream_df["label"] = labels
stream_df["stream_type"] = "online_same_domain_stream"

stream_csv = BENCH_DIR / "same_domain_stream_benchmark_with_ocr.csv"
stream_jsonl = BENCH_DIR / "same_domain_stream_benchmark_with_ocr.jsonl"

stream_df.to_csv(stream_csv, index=False, encoding="utf-8")
with open(stream_jsonl, "w", encoding="utf-8") as f:
    for row in stream_df.to_dict(orient="records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(stream_csv)
print(stream_jsonl)
stream_df.head(20)

arxiv_ocr_benchmark_workspace\benchmark\same_domain_stream_benchmark_with_ocr.csv
arxiv_ocr_benchmark_workspace\benchmark\same_domain_stream_benchmark_with_ocr.jsonl


,doc_id,parser_name,text_sha1,char_count,primary_category,title,label,stream_type
0,2604.00977v1,pypdf,5bd3ecacae1e2f004359ae6027a87486a1d24035,49999,cs.LG,Flow-based Policy With Distributional Reinforc...,0,online_same_domain_stream
1,2604.00112v1,pypdf,097928da185b0933c0214d4c4ffe344bea6dc5b1,42736,cs.CR,Efficient Software Vulnerability Detection Usi...,0,online_same_domain_stream
2,2604.00731v1,pypdf,198ded00ddee3132b243047f0e907f90b293a872,42490,cs.IR,STCALIR: Semi-Synthetic Test Collection for Al...,0,online_same_domain_stream
3,2604.00316v1,pypdf,e577423b954a505807374c97c82c3dda06b4b597,70908,stat.ML,Breaking Data Symmetry is Needed For Generaliz...,0,online_same_domain_stream
4,2511.16528v1,pypdf,be65a528f27f46d4e4c978335fe7e10c04262165,34160,cs.CL,TurkColBERT: A Benchmark of Dense and Late-Int...,0,online_same_domain_stream
5,2604.01073v1,pypdf,ab1fa704894564fc3909f71c7891684c9a8cdfa3,35180,cs.CL,Narrative Fingerprints: Multi-Scale Author Ide...,0,online_same_domain_stream
6,2603.29665v1,pypdf,ccb2db7a2e4dbaf9e72f18fded1753bfdb89e255,43747,cs.CL,Near-Miss: Latent Policy Failure Detection in ...,0,online_same_domain_stream
7,2603.29068v2,pymupdf,a005798153ccba81f774f09998ad6c9ab7cf2dc6,46757,cs.LG,ARCS: Autoregressive Circuit Synthesis with To...,0,online_same_domain_stream
8,2604.00754v1,pymupdf,fb2bfe75470d3a589317354a966dedf3c00e543b,47504,cs.CL,Stochastic Attention: Connectome-Inspired Rand...,0,online_same_domain_stream
9,2603.24318v1,pymupdf,01ceeab6e2d5c3babfa595576fa21eb27a065d4e,271996,cs.RO,Toward Generalist Neural Motion Planners for R...,0,online_same_domain_stream


## 19. Gợi ý dùng thực tế

Thiết lập hợp lý để bắt đầu:

- **Laptop / CPU**:
  - PyMuPDF
  - pypdf
  - pdfplumber
  - Tesseract OCR

- **1 GPU khá**:
  - thêm **LightOnOCR-2-1B**

- **GPU mạnh hơn**:
  - thêm **Qwen2.5-VL-7B-Instruct**

Nếu mục tiêu là benchmark giống paper hơn, bạn có thể dùng:
- parser nhẹ + OCR cổ điển + OCR model
để tạo nhiều version text của cùng PDF.
